# Практическое задание 3. Линейная и логистическая регрессия

## 1. Линейная регрессия

**Задача:** предсказать цену бриллианта (`price`) по его характеристикам.

**Датасет:** `diamonds` из библиотеки `seaborn` — содержит ~54 000 записей о бриллиантах.  
Признаки: `carat`, `cut`, `color`, `clarity`, `depth`, `table`, `x`, `y`, `z`.

**Алгоритм предобработки:**
- Проверка пропущенных значений (`df.isnull().sum()`)
- Анализ выбросов через `df.boxplot()`
- Кодирование категориальных признаков (`cut`, `color`, `clarity`) через `pd.get_dummies`
- Удаление выбросов по IQR для столбца `carat`

### 1.1 Загрузка данных и первичный анализ

In [ ]:
import seaborn as sns
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score

df = sns.load_dataset('diamonds')
print(df.shape)
df.head()

### 1.2 Проверка пропущенных значений

In [ ]:
print(df.isnull().sum())

Пропущенные значения отсутствуют. Датасет полный — дополнительное заполнение не требуется.

### 1.3 Анализ выбросов (до обработки)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
df.boxplot(column=['carat'], ax=axes[0])
df.boxplot(column=['depth'], ax=axes[1])
df.boxplot(column=['table'], ax=axes[2])
plt.suptitle('Boxplot признаков до удаления выбросов')
plt.tight_layout()
plt.show()

### 1.4 Удаление выбросов по IQR (столбец `carat`)

In [ ]:
Q1 = df['carat'].quantile(0.25)
Q3 = df['carat'].quantile(0.75)
IQR = Q3 - Q1
df = df[(df['carat'] >= Q1 - 1.5 * IQR) & (df['carat'] <= Q3 + 1.5 * IQR)]
print(f'Размер датасета после удаления выбросов: {df.shape}')

### 1.5 Анализ выбросов (после обработки)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
df.boxplot(column=['carat'], ax=axes[0])
df.boxplot(column=['depth'], ax=axes[1])
df.boxplot(column=['table'], ax=axes[2])
plt.suptitle('Boxplot признаков после удаления выбросов')
plt.tight_layout()
plt.show()

### 1.6 Кодирование категориальных признаков

In [ ]:
df_encoded = pd.get_dummies(df, columns=['cut', 'color', 'clarity'], drop_first=True)
df_encoded.head()

### 1.7 Обучение модели линейной регрессии

In [ ]:
X = df_encoded.drop(columns=['price'])
y = df_encoded['price']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

model_lr = LinearRegression()
model_lr.fit(X_train, y_train)
print(model_lr)

### 1.8 Оценка качества модели

In [ ]:
y_pred = model_lr.predict(X_test)

mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
r2 = r2_score(y_test, y_pred)

print(f'MSE:  {mse:.2f}')
print(f'RMSE: {rmse:.2f}')
print(f'R²:   {r2:.4f}')

**Интерпретация метрик:**
- **MSE / RMSE** — средняя квадратичная ошибка; чем меньше, тем точнее модель.
- **R²** — коэффициент детерминации; значение близкое к 1 означает высокое качество.

### 1.9 Сравнение реальных и предсказанных значений

In [ ]:
results_lr = pd.DataFrame({'Actual': y_test.values, 'Predicted': y_pred.round(2)}, index=y_test.index)
print(results_lr.head(10).to_string())

### 1.10 Визуализация: реальные vs предсказанные значения

In [ ]:
plt.figure(figsize=(8, 5))
plt.scatter(y_test, y_pred, alpha=0.3, color='steelblue', s=10)
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', lw=2, label='Идеальная линия')
plt.xlabel('Реальная цена')
plt.ylabel('Предсказанная цена')
plt.title('Линейная регрессия: Actual vs Predicted (diamonds)')
plt.legend()
plt.tight_layout()
plt.show()

## 2. Логистическая регрессия

**Задача:** классификация опухоли — злокачественная (1) или доброкачественная (0).

**Датасет:** `Breast Cancer Wisconsin` из `sklearn.datasets` — 569 записей, 30 числовых признаков (радиус, текстура, периметр и др.).

**Алгоритм предобработки:**
- Анализ пропущенных значений
- Масштабирование признаков через `StandardScaler`
- Разбивка на обучающую и тестовую выборки

### 2.1 Загрузка данных и первичный анализ

In [ ]:
from sklearn.datasets import load_breast_cancer
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (accuracy_score, classification_report,
                              confusion_matrix, roc_auc_score, roc_curve)

data = load_breast_cancer()
df_bc = pd.DataFrame(data.data, columns=data.feature_names)
df_bc['target'] = data.target  # 1 = benign (доброкачественная), 0 = malignant (злокачественная)
print(df_bc.shape)
df_bc.head()

### 2.2 Проверка пропущенных значений

In [ ]:
print(df_bc.isnull().sum())

Пропущенные значения отсутствуют. Датасет полный.

### 2.3 Распределение классов

In [ ]:
print(df_bc['target'].value_counts())
df_bc['target'].value_counts().plot(kind='bar', color=['salmon', 'steelblue'],
                                     title='Распределение классов (0=злокач., 1=добро.)')
plt.xlabel('Класс')
plt.ylabel('Количество')
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

### 2.4 Анализ признаков (корреляционная матрица, топ-10)

In [ ]:
top_features = df_bc.corr()['target'].abs().sort_values(ascending=False)[1:11].index
plt.figure(figsize=(10, 8))
sns.heatmap(df_bc[list(top_features) + ['target']].corr(), annot=True, fmt='.2f',
            cmap='coolwarm', linewidths=0.5)
plt.title('Корреляционная матрица (топ-10 признаков)')
plt.tight_layout()
plt.show()

### 2.5 Масштабирование и обучение модели

In [ ]:
X_bc = df_bc.drop(columns=['target'])
y_bc = df_bc['target']

X_bc_train, X_bc_test, y_bc_train, y_bc_test = train_test_split(
    X_bc, y_bc, test_size=0.2, random_state=42, stratify=y_bc)

scaler = StandardScaler()
X_bc_train_sc = scaler.fit_transform(X_bc_train)
X_bc_test_sc  = scaler.transform(X_bc_test)

model_log = LogisticRegression(max_iter=1000, random_state=42)
model_log.fit(X_bc_train_sc, y_bc_train)
print(model_log)

### 2.6 Оценка качества модели

In [ ]:
y_bc_pred = model_log.predict(X_bc_test_sc)

acc = accuracy_score(y_bc_test, y_bc_pred)
roc = roc_auc_score(y_bc_test, model_log.predict_proba(X_bc_test_sc)[:, 1])

print(f'Accuracy: {acc:.4f}')
print(f'ROC-AUC:  {roc:.4f}')
print()
print(classification_report(y_bc_test, y_bc_pred,
      target_names=['Злокачественная (0)', 'Доброкачественная (1)']))

**Интерпретация:**
- **Accuracy** — доля правильно классифицированных наблюдений.
- **ROC-AUC** — площадь под ROC-кривой; значение 1.0 — идеальный классификатор.
- **Precision / Recall / F1** — метрики для оценки по каждому классу отдельно.

### 2.7 Матрица ошибок (Confusion Matrix)

In [ ]:
cm = confusion_matrix(y_bc_test, y_bc_pred)
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Злокач. (0)', 'Добро. (1)'],
            yticklabels=['Злокач. (0)', 'Добро. (1)'])
plt.xlabel('Предсказанный класс')
plt.ylabel('Реальный класс')
plt.title('Confusion Matrix — Логистическая регрессия')
plt.tight_layout()
plt.show()

### 2.8 ROC-кривая

In [ ]:
fpr, tpr, _ = roc_curve(y_bc_test, model_log.predict_proba(X_bc_test_sc)[:, 1])
plt.figure(figsize=(7, 5))
plt.plot(fpr, tpr, color='darkorange', lw=2, label=f'ROC-кривая (AUC = {roc:.4f})')
plt.plot([0, 1], [0, 1], color='navy', lw=1.5, linestyle='--', label='Случайный классификатор')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC-кривая — Логистическая регрессия (Breast Cancer)')
plt.legend(loc='lower right')
plt.tight_layout()
plt.show()

### 2.9 Сравнение реальных и предсказанных меток

In [ ]:
results_log = pd.DataFrame({'Actual': y_bc_test.values, 'Predicted': y_bc_pred}, index=y_bc_test.index)
print(results_log.head(15).to_string())

### 2.10 Анализ неверных предсказаний

In [ ]:
errors = results_log[results_log['Actual'] != results_log['Predicted']]
print(f'Количество ошибочных предсказаний: {len(errors)}')
print(errors.to_string())